# Chaldene Slider Demo

A Python `interact` slider controls the lower threshold bound of a VP cell.
`ChaldeneClient` communicates with the extension over a Jupyter comm channel,
so `set_input` and `run` are ordinary Python calls — no `display(Javascript)` needed.

> Run cells in order with Shift+Enter.

In [2]:
from PIL import Image
import numpy as np

arr = np.random.default_rng(42).integers(0, 256, (256, 256), dtype=np.uint8)
Image.fromarray(arr, mode='L').save('sample.png')
print('sample.png ready (256x256 grayscale)')


sample.png ready (256x256 grayscale)


---
## Step 1 — The VP pipeline

Run this cell to open the visual canvas.

In [2]:
{"nodes":[{"id":"0","type":"read_image","position":{"x":100,"y":150},"selected":false,"data":{"specName":"read_image","displayLabel":"read image","description":"Reads a JPEG or PNG image from disk.","inputs":[{"id":"in0","name":"path","type":"string","displayLabel":"file","description":"Path to image.","defaultValue":"sample.png","widget":{"type":"FileInputFromServer","extensions":[".jpg",".jpeg",".png"]}},{"id":"in1","name":"mode","displayLabel":"mode","description":"Colour mode.","defaultValue":"GRAY","widget":{"type":"Dropdown","options":["GRAY","RGB"]}}],"outputs":[{"id":"out0","name":"image","type":"image","displayLabel":"image","widget":{"value":{"imageUrl":"...","dimensions":{"width":256,"height":256}}}}]},"measured":{"width":301,"height":136}},{"id":"1","type":"read_image","position":{"x":550,"y":150},"selected":false,"data":{"specName":"threshold","displayLabel":"threshold","description":"Binarizes by keeping pixels within [lower, upper].","inputs":[{"id":"in0","name":"image","type":"image","displayLabel":"grayscale image","description":"Input image for thresholding."},{"id":"in1","name":"range","type":"tuple2","displayLabel":"Range","description":"Lower and upper threshold bounds.","defaultValue":[0,0.9],"widget":{"type":"HistogramRange","min":0,"max":1,"step":0.01,"value":{"histogram":{"type":"grayscale","data":[0.4540636042402827,0.49823321554770317,0.42579505300353354,0.46819787985865724,0.44346289752650175,0.44876325088339225,0.48056537102473496,0.4081272084805654,0.4169611307420495,0.47703180212014135,0.4628975265017668,0.48586572438162545,0.4734982332155477,0.49469964664310956,0.42579505300353354,0.4204946996466431,0.43462897526501765,0.45936395759717313,0.44346289752650175,0.4717314487632509,0.4416961130742049,0.4628975265017668,0.5123674911660777,0.4734982332155477,0.5070671378091873,0.450530035335689,0.45936395759717313,0.4469964664310954,0.43992932862190814,0.4452296819787986,0.48586572438162545,0.4823321554770318,0.9699646643109541,0,0.46996466431095407,0.48056537102473496,0.9876325088339223,0,0.4169611307420495,0.46996466431095407,0.9717314487632509,0,0.44346289752650175,0.39045936395759717,0.9222614840989399,0,0.4204946996466431,0.450530035335689,0.9222614840989399,0,0.39045936395759717,0.42402826855123676,1,0,0.4558303886925795,0.4469964664310954,0.9399293286219081,0,0.43109540636042404,0.4452296819787986,0.8886925795053003,0,0.5141342756183745,0.4558303886925795,0.4293286219081272,0.941696113074205,0,0.4664310954063604,0.4558303886925795,0.43462897526501765,0.4575971731448763,0.43286219081272087,0.43286219081272087,0.8798586572438163,0,0.4734982332155477,0.43462897526501765,0.46113074204946997,0.4416961130742049,0.43286219081272087,0.42579505300353354,0.9575971731448764,0,0.4787985865724382,0.4381625441696113,0.4558303886925795,0.4540636042402827,0.5371024734982333,0.48939929328621906,0.8939929328621908,0,0.4540636042402827,0.46466431095406363,0.44876325088339225,0.4823321554770318,0.4293286219081272,0.4081272084805654,0.8851590106007067,0,0.48939929328621906,0.44876325088339225,0.4381625441696113,0.4204946996466431,0.43286219081272087,0.43286219081272087,0.950530035335689,0,0.43992932862190814,0.4752650176678445,0.4275618374558304,0.4381625441696113,0.4664310954063604,0.4575971731448763,0.9487632508833922,0,0.48056537102473496,0.47703180212014135,0.4098939929328622,0.46466431095406363,0.4469964664310954,0.4558303886925795,0.9151943462897526,0,0.4204946996466431,0.3833922261484099,0.4628975265017668,0.44346289752650175,0.4664310954063604,0.4045936395759717,0.49469964664310956,0.43462897526501765,0.8515901060070671,0,0.49646643109540634,0.41519434628975266,0.5017667844522968,0.5,0.450530035335689,0.45229681978798586,0.4416961130742049,0.42402826855123676,0.508833922261484,0.4204946996466431,0.4028268551236749,0.5106007067137809,0.43992932862190814,0.450530035335689,0.8639575971731449,0,0.46996466431095407,0.4540636042402827,0.4363957597173145,0.46466431095406363,0.4363957597173145,0.4628975265017668,0.4575971731448763,0.4363957597173145,0.4452296819787986,0.4452296819787986,0.4381625441696113,0.46996466431095407,0.44876325088339225,0.39399293286219084,0.8515901060070671,0,0.38162544169611307,0.4558303886925795,0.4664310954063604,0.49646643109540634,0.48056537102473496,0.43992932862190814,0.43286219081272087,0.4169611307420495,0.4752650176678445,0.46819787985865724,0.411660777385159,0.44346289752650175,0.4540636042402827,0.41519434628975266,0.9028268551236749,0,0.4840989399293286,0.46466431095406363,0.4911660777385159,0.44876325088339225,0.4540636042402827,0.4381625441696113,0.42579505300353354,0.5070671378091873,0.4416961130742049,0.4363957597173145,0.4540636042402827,0.4575971731448763,0.4293286219081272,0.43462897526501765,0.9346289752650176,0,0.46996466431095407,0.43462897526501765,0.4293286219081272,0.46819787985865724,0.4717314487632509,0.4734982332155477,0.43462897526501765,0.4840989399293286,0.4469964664310954,0.45229681978798586,0.43462897526501765,0.4628975265017668,0.46466431095406363,0.42579505300353354,0.9045936395759717,0,0.47703180212014135,0.4840989399293286,0.48586572438162545,0.4293286219081272,0.44876325088339225,0.46996466431095407,0.42579505300353354,0.4363957597173145,0.4363957597173145,0.43992932862190814,0.41519434628975266,0.4787985865724382,0.44876325088339225,0.44346289752650175,0.8568904593639576,0,0.46113074204946997,0.41519434628975266,0.4169611307420495,0.43109540636042404,0.4204946996466431,0.43462897526501765,0.4752650176678445,0.4540636042402827,0.42226148409893993,0.46113074204946997,0.4416961130742049,0.43462897526501765,0.40636042402826855,0.39399293286219084,0.8639575971731449,0,0.4929328621908127,0.4275618374558304,0.43992932862190814,0.40636042402826855,0.4416961130742049,0.4381625441696113,0.4540636042402827,0.46819787985865724,0.5035335689045937,0.45936395759717313,0.4876325088339223]}}}}],"outputs":[{"id":"out0","name":"image","type":"binary image","displayLabel":"image","widget":{"value":{"imageUrl":"data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAQAAAAEACAAAAAB5Gfe6AAAlQUlEQVR4nLVdibbsKAhs/f9/zpyOAlVQmPQ9b3Jm3k0nLqhQLC4Z1+fz+Yxr/S8ufDyuLpVIWR8eLkiWKjmRJd/lh+OTU1Hx8/7zfXJ9Rikr/b5WSk4wvv+OVEBqfyk5P7zaSrxULuNSLctFLQoyyYumu7jxuSYUXspb/QJZnZSREqxUnnYYtfc4Men+K1LzO9HthbrV4emyjAOrzKkupPX6fIID1Dh5m79/V9b7xopd7wvPUa+P6wqBiGGDyiKxUTU0EZFREBwlX5+7/tSvkg3HZ/p4js/lSWgAuZE0brs7owhIkeoLnllJo09EUuoDG7NMAHbDGhZOAIXcHHFtxohWjs/1mfd47ibuLHdXOMFefyIVJAmyxvggrC36vvXdd/ZPItkq2AlS7+X/qcNjpMr1LWtz5PpzU+J0jh/hupb/ANonzWEQSX/s5qRvNPx1ScYtg7q8sTAgGEbggJYduL4jhkKzEphc0WAyom82AJS7sy1RrBjnDJxE3IpEob+CiTcPm4BHK79ZrgkoV7p9EaIQMvj/rkjkkoBmxVdEoWxCP5CmsYoxjXW0d17gE5dGkHudTCC6UhrPZs+fy+hTeEGamCVEKvtfxTZyj1sNBkNJFAEtUEcEIBfxFX+BzUAPXHAWBm/gM2JchvbYKhFdIlR0lxZl+fAKFY398sZeTbao4BCZ7VYJtVZL99OYvrCYsejViSyt1zaE+oKwe0kDm+r0B4RCarhN6opBDde31JeDly3mki7p2a8u3GiF7D6Vjc1VO3a41V9ou/F06fssAGR6ZuOstqcojVXYLlqLKCYvZlKtwkyiLVfXysq3WCizeq4ydLcEKpFRaXnSI/TqUH2xBgKQd64HgVqpailtwS8llGW+e9ehfs2NiiaTInuTFdMDqI3jAOtMP+cp5EfHH5Uj53lh/6kaThr28xWuvgj+/Ya/T0bnT8GMF917YJ1TIfnhtOf2IBt1jfhh0tC7yZBrrp6BkoWImKk8RnrAemijXKL41jPfsjbJIzogvLMohhwK1XSojrw7cx+InNZdsxYmwylpWZAElfurQNEaXgY8uA9RlDm4pAX2dfpV30iGzbZxhqkn0GPRzbdn26fVOYdrbF1xcEFBZF/pFbc3jkRk/dvrjl11UyYPb19R92pu0wesler5hTzd8tHY2juQ8+WsNZQgMNV3vdKth4rCqtwlbNKaPrVgR3sdXl80EMIAOPK2SNJnPr2GwXQ2a2WRaYQcD0ONZhqJ2nitzqmyU4MOTxgMvEu/EIYS0Fh+Aj1yPOqFqZBKm5G3QVi7LK52ChCpAK7fljjPBVxMCKCVj2rbChWEjpNo09xDBwzzwyQGJIWsqDPtzwFcI2cXn7x3iFlRTfY8+gu1KninqzyDnN7TwrfZyhkv7ClxCaVWsb8xxo+WZq8tKQO9zDY2G6OP7ZnjlUOu+tTu3PHNdcm6SyJzoVdIKA3jHvwvUbcFdzdwW137VYLMZIwSQKhrpkhGE65IyE9RqCQ3pZ5GLlk0rVrHBVLMN5/HbQTP9l2qm2xi/6fK7veaXZPTxQX6SCXNAjjSlBDDZmVYW8xEL9kyIoTWzMlv1qC5u21fIKOmqPOQ4lwlrdBV5PuV7xU1U0bp2rzShfyOwUGMa8C8VTDtQXDyeQiZn779S8yQIUUXQXjxJaXCB6v3TCW9cBC68cD/u2OAKWloXns+zRCBql1tQXab3atjUIhnHDMaixWA3UHPjNSwuQMYZEZid7dWLM5uVBctaS+vX4zZRom1Wsfev7fQXpBCzlN6AY/9V9PA9uXoiX12Ll2SnEj0997o+dcX11XJfFfFfJTbWq/4jfZdFOL/2A0bEK1uzsLdWHnAXaD/smuLhQpVPbuoF1oiFCFyUykmDLYeMyzhLqHZ5eRxkO2N6RYCrIHkNjrOAQ67Tg27WHVtGOqQZhACdTLfSiCmqm5sLShLQ+PhlUxqkN6xeMWKBO9Xqul3hxId2762aJrdMUx4Wo5LaPn+1vg471Eyy5+jxQvVUechfoueoudUcgxZtAYVl+sFEhfVToY885ZSYcR8F3cBFoH0ZICyFycjOGlwk+fd0HItuYa2OIEJxlL+r5SbMQxPETGoxz5hLeComEWR7NwXPKendt5ZAY0yg7sjFzZK8ZF1pukM792x/C5bPLau9cRQGQviMSWN6Nnqyr/kC1oVVBBZp5YCuRF48bsoiBw2FYHKFXBPlB/pUacPUMggTRktbQOdB+gvJpkGvpa9Bi2QSJPwu7dDW5IwG88Zh7ud4yhPwriDGBGDAr86MZVb7jL+aMV1LzjPZlrBu4YBV+mWJPpJVSmeOk4YFyTpyjldFdVFmYecplZqMHz07XqICLwit39VmVLYrj/Xj8ruuVu+114tjnowBXvwQhHJYILwtX6XFOS69qEz7SZLccgGtwmqq84Ezh46SEHRa70kEqr2Nv0s14inJc3fLi0piBZqHQVUSrW+VD0PEjwMJZBWWtc77pLLlXxJkmUNpVkl7nnugZc77fMkAsVgfKpMiNkMJc8jmM2vPaLY9eMwvnSXbJNCl3l4MLrk4sqmmM/4tViJgE128TA9B5QwfNqQdTpVxbtTHoemt8SCy7iQRntUJjt6rrWUJwVhJCmKi8+QNKq0NvrbhkLo8WO/ypfVB3VaWSbO6mz6RpiOt/2xRSYkYGP2eltQ0yWJSsxN9H9T9m1i2ZMkYCbLEJwKeq1Ml84Z7QvZYEBYRt9a4qpbTju7pMxmb8DRu+nv+6V3TUR5FiBZ28lmddrPHscGD9ORV5jC3nHRU5YnUxn1bFAxFltwqhukgBD5iVT6LjO1Ia8zTqyPu56q5xvxftLqs4zASvZdTQb2P68nBJWsQkgnNQR/gBJncbsxT/KuN+G5djzvBrPiBhpXaasDv9Or1GufAjYdIpW4u27lC4gvr7DWPVcW5ulTWdLTfGlHz6JZ0jxtxAnufzdTLEOPZomFF7ZvuiXwya7yWglscBWUQiBwWZk3naNaYFruaVFrItx4dr7fdbVMJXzRd4V0I13Vt9LFyL5TjMV3NwTgalVgKHtqv6W6KjojfEg1uapijyBnTIjjHWS/A7/BXkQonVndQsZSYqHbNDmiCjU7WJB21hWtnxSB6V7yseqNuLY0iF7ZIpFHa8b7vMcvBgsWM9Y9e3cQdZdhmoOcgAVraHAVLVC64bjuKcFSVZaQZetUGzujItzHsR4UKKxCdpbXM0yQR3GU9bygsZ0Aych/MPzTCgwGvFlMBlw6C52jqeiNASjO/sQqnzSEjCo0+JuFMJkLs42uSEg50ko6ZOEr7xpLnuXuvftZwCK1tzP8uFDk5b2VmwotJAQZTvpiWxjLb6tI4sTF5AklMtnOIopYt4YFtCK+0Pe5HWl4I9ZCPgxSRAxaNA8iJw7OzlAXY2OfZz2YMHKy1RNmP8AE5cdh8dqAq50/oJdjzMZ3ud/xEr5banEq1vYiYhIRkLOXd3eBkh8Nbmi8CqTxO8Yoz/fWlKJshwQJ8l4kBKs9hQ6oiGkUJFVSBpodj405laVtAS9ykiQ0yWZrr660MSf7gDnAi0YCnAHBILIWfzx0PVNSEh/ioFW1PlaXI29v60qvU6TINxTRa0s0dxJLmikiSPOtHHDCBqeFC/1Yt8IKXgFejoCznZAROW0Ly7RWyDH0/rrBNNGLODiuP4ilMo0oyPjkt77yWjnxM6icVobGj2wwjScRyLDRRCJlxr6j/kGed57kDiz0xU1I2ovCVVHmks41mQcGXUlnNii2JQStjjZtIeB2X9mF3UgYq8ubki68TZ5Bv7ijWN5igcYzcx+vJ5dBOiRKxkTzyRL0a68JAUWz91YWn2wBEg9Bten6BrjZVbw8+Fup7idr/KnzJ3gpDoDhm2NlV+vnnDXWAyZJqOgXt4pnqgTDw/3XrbW16++EgbkU9/EmZ4i/3X5EClq4lLrGtcseULAk2ESgDY+nmjEIE30bY57lvsl7X1wV3ziUavIlaJNiWsDHnb3lvopBFgNkIUAzEgoM7gm5W47LfVHvpxl4pAic9jBS3BfGGqEqR+qomewa4C++cJ1mH4OQj46ZMoj9ui4v1VyQisTd2jd+Pbvm5dEc6e4XfGGz9ZCWi+eWRHc364xtFeMoB450YCSfPNj2fxlCL/elQ1k04nuPZdjucRIvwDMRmfiaOlcLfhwJCWeES+h+gJYS4IFn9BB9uXxE0xrhygGimQvdRSdFi+GfTdzWRAfHgvb3IX3oXnEfWHrVS9W2MaQkJ5kMEZqX8niI/RyPPLwT/sDHer0ovW+Ms93y13W9kNmA1A58x4oKt5GIxWM0JCumdQpKBM52L7HEzct7hfJDX6Mtl2Q2+JRYYP/dvEJGzHbwL38SA4dV9hjsI2ZK8T3mZX17rpTcWG22wstfcHdgWPyqkobL7+q4wxDwTt9SDVv5eW6tnjm54QUrupuFCEHhkvsF7RMUDEhI5/kmG7c81YBMWeAsUSn0RcbI/RgyBk0Vw65+Jd2iJQWNYuRD+5MbxyNsYzH+qKhzpeXgKw05Bx3dWBfCFC0GkLJ/qkVbst4/iAPagAg/inEreJN9Dr5u84uLhXFz1U5U+jq6bHxwmpUEUdC8IFp5H9v6nIev3NI8hg+rxg/WfdZC3Xv7GQ6NTFVHryv0F7aeGTOu3JW+RrZyynqdvU14ZX/B3olxLqZegWEgqCsbSo1Cm2U5sgET7Kn1yrdRSMwXPFh4SAqT90Zj6fYX231cLA7b+n9PvsWq3EgW6Bt3g4/RMehHXchNqFSAcKW0pYTdGAoALLtKrHnYNxChXD9Dr7A+3WVedwgscm2bPs9kWLlXhU1xte7fk9UEm1REoQXddVKIERQ8Ks/y7aNdN96wolQuqtd+6MnHTFRpg9XK7H+DgZBi1pfw776x49TgywoEbC73yK1cYPFL4wr3kIXKmlNsqWhBUcMv0QXYZCaEL9C5azLs2N2gUCUcdh6CJZIZlotjLtRAaMBMHJ7nHmkKgODLdJlROVdxYY6Aky0y7tEoKoFXZKT1vbbGmbtFKY9oQhTdbLXjCCmcnYPLUSB1rsvyz9I1kck3OSZlRoeuQKuZPgBr1f/ca0nJ6I9KJGJyA0MMjT8bcCERGKjxHg2+9nlgz/FUllrXH30RZf0/1FwB875wagx7WGnvVCCa7YA90fGH5UA+XdVTDCajyJ8YYO9mOfUmzfjFNdvk3Q+ja8cGedUTcRGIubZlldOFUHCfnJY5P1XkkFV3s+Tek90zC4UOIbZBRRyCBOjMpRap5x2diURB0VpktPu2BKsxONtfvUocfzRNhnQI3RY5Gx/N3MYhMvCHKw4NPbq36Fru22kP3NZO3ebrFlTvhdfRE88LdGgIe3dL6X71NnQxzgFWgwoLjAzfy0+W3u1IyhX+ypnPhjz+adOvxaTbbM2Ai6Jc4+fNnutyLDUerR0HHBRrZJydm3Tl0ERnjze/D6ETSlKzPRScXrrpxKLZCPHItbXC7q9fyWrj6b0xEZDk8i2TYhaqT4e8u1bzp/2CgFtaLoVrZr6ZlJSUC/TEItLVVc6Yi0hCsrk3pJjUX3eyNBcaTgpb6J8X57uxU/rkaOkoXz7pESeqDg46b9Lj4kUE4Mi6ggxlCI3qxy4iwwlOtJ6vPRMJ+1TQDdFUU2cbeKZRJ42BWv9AVmcoT9RLtsoKTZ+8NCWZVszK4pwpyHrDMLxBzwUSobCgAGDZ20jTDCmYipxU7oALjb5dAJ5XYbadKDNVhuE6fMxmXxk71bkwmeZD4Ubx6rh0oIOizZdN2UhiFrN65l3ufQZXaipp7f1OTDBFtDF2jlB/5m1mmkEXeYHEgXGWCfgfuAx4juKediu4BMyD68sBTTwX2wJtLM2grey+hGxX7JQmQcBBsafBp1i7kXowNi+RYsMsfToy23Ur9fRmpsbFVsBqjh3mYQOS9qFRxIGW34cKlC/IRvTk/pXQOZpcqw77N74wGfy13yFDXvVOXYfzOx6NkBd1nK1EaQmyFt2/XlCWjKrZ8H7GIzNFIEYrcFALewxV1JGxv18YXjTcsnWXecYglSMisj+hneMXG9I6XJovbqgcrKnMCd36vdShaYhVuS9nYjTdM/d7Nw5G8u5qBipXDbz7yZhpJ2cc318C4mpRuaHhu+1x/PyKpPTeHWMFGoAHoZlZrxdi9+j9qC1ySaRxB0JTzfERPIW/DVkHahUYTU5x/3OPKdgvBcXTRfhdfW7fTFfQwxeT+4ji5CmRdYs9BGeN0zMv8CbOSihbZR8fv1/dydebe4DPzkGwFGz4aXGiLNIQDuZPudCH+MSg193285Tl0KJI4thbCjoJiA7CyAplNKKjemKq+NdnA4UfKNdfLCjMtgi7Su6DB5Ry6IGsq3QokuOsgBQZPHvX/g98cxSEJ7sqWaRIzAGt0YzzMniw2aWGs2tJpS87PYyRwispD38tL6WSX6qODrgIScSC1XQOatyVpaloCFt/OmaVBrSGlB/h5CxS7BuwZkJFL6L8HA9Hk5VExbVHMYZQNn0N6zu48xJaPdYXcCz6taosKTLMNRGhAXP+OP5pDSuX32pF/IIYGUt1mxtkMmapfCzVAUKfPsIGjtNGOREtuXQ78uElpz7/YRMNP/pR+Qr3h+66zXMqncPxtJdpYGDTPZmkXKKJWvIwKQ2sP87ec1EUymeOrMGeQwLgDuoxQ1SD/U7nqmhG/I/JYzLhCdC1SimRQSJ8bV7nkkJrZAcBL1YXRJ0Pjf1GczKOwfJWigE00Zmhv46fbqRgmJJhrEhAPqF+aF7aGNjwHJgViOve+Ti4y8hYznWULEFiVz5S3zxAbsVykslOQB/d2A6/6WQ91Eav9cYTJgwg+ydMEtbm20KgIat1mWAuo3M2n2ooHVWM/7E2TCA4QOqMWImznRmLsVxgIP8k+yTHceqHx26GJgXHS9iy87iABaaE+QIxu9Snt99cZ5ddvj6N259oeCr9rLfjfsL0fMG2ztpxCC9PSYdgwcqoArC29L06Ob4QpQNDBgiHoe1PJx5eCeW39UMVnf4rG7azakBlxr6CsNqqI8a9G6Vqmrdp4PE8Mw/3eE1w5FD7sNHBtGBbnZhsdhbyixHg95Y9gd0uumFlfAx+HBxQLkwgLNlY4cI3z0eLt9ebTYPihPoN1Y9xpR/0aOCszsRYNTEJxv+sl5ZlYQVLb8VzJzQuv3jmK8zR/G2C+x9CkBAIMpKyAW+USEbMym51xCzt2pFHY+mhp5hDE4FSLy43dBPR1Ah4AQL8CXweHSOkDoUuBmed8GMdRmJ9pQJaE0LF3B7DesKwOaWm52gwxQOZp7XlVhbYnWsyZ1IykZvoKiYUWxDJYBe5VmIYpN7PApbfxd61oLF0l2WTuCafDnkafH1jEMy3mMzZDogR3Bj9/XAdUvbjvQMMNhjKbsmA/O6KI1P1AawPCDv97hKeZbFnKGWqSGuzldFWYTgQXgdGsRl52Kh/MEvio7XfZQkIEXwWP7UhSp1sf2EbzvZMFJH6ZT9mrImpIQ17pFTMy7X16OyDc8EA74j0q5R+eYhxzKfOeKh5PIP4jIJi63/K23Nf4Xd1jktn4HugP9OoljGcaRkArEdIzweVPQQO2uvQJbjf86gelI76q9aZocoTD1eTlS44royS4HpM5RgXNQEylAJLp2v3E9okpYkVPoSyGPTMuu+HEImMMqh7FKVsAPdDFrz+luUaTu/6dNKv7JTtfvZC+K9n8RAFllHzhxaAca7mBA1ZrJtMZ6URN+lR1jOYybub+Pzmx7K7wQAPu6Vxfg/GESrTxfduiqRWXIJ2K5jPfaveKZi7fBr4MnxYq4E3qNEOmk+X8rKiHIPiFyEafeCx8j+LvdaUKN60mI3e4IUxN2A2sVR2wwYwE4//4lux0U5Q4GuKyNa+BSVxd17bAazGzJLRO4zEeOSLX4aJQFkDaszm2Yo+yMy05lndKgXbBC9hBUVz9Uo5ryOx5RbPtOhjr6kUx/473sJs+bWjwixtDICJ8D0LrUzkLBhqVQL/Tm/5O7fiEj2UZvMBYdkcwDspXONR3712TjjxacddxY+H4gqxemmowjAFNvcV3xrjKvnutAAlWUNJ58poMFq0SxE0Iy+1cjucWV7BSorvROTWTu1oRRn43epcUzeTkRwXzbe4J6T0UzHi8idM1Q/BadAe6GbMPBFWQa9GmWJpSbJJasWmypflwwAbmLFGaPNA7VVSLfmLmIS/iAGbePPAC1lR+v7nehJvlqhnK/35Ojpgx/qlbZGbkO/PTZy5TWK4sbyYxQE+GQ/nx9EtK0rWpJUOpo6P2QWJBl6xnl0ZUuS0yvuUbILpqp29uBNkvxxp3HwNUsquVOjQRFa5Tl5NbcTEqNTl0ZURpqArOnQbQwRJFoAinyDvdckA5pQp2yD5BKtm8phSyMgKrWMTZLlvLUAeh2sGWqUaNudsLJFRA7b4xOaa1Yk2D2cM67RDiYi2928xJJ7JsVVpsMwecNTcpQ2V3n14AVTFsPhUM1w8VHkKWMGy4iMBNRr8BK3iQ0sXvEat8mQDkWGRvJ897qzfmC3TF8vDQoyuEjyVWB8CK9yCzrBG12gqt2Yl2i0sNglVDg0R57fdDCj6z+fY4MQVK5D5vPQqe25eCchUoAP0AwFMLKYbP8UPT9cf9LncR9LHIhIrPrsO5Bi00jwVco8nLqr6v7El6C7qWVxHogZvnAxSLGWfvJsiuTpCa4buylITf4R+Y71brflEuFY1rJz08eX3Tbaqd8OilmxW7XVQ8F22XF2hJO450RQYVrVZBaKQrPhGMjVjU5Vql5o7A+Mdv6vaNoc3PBaw96h2FwJrTTZKWx+Vnv7V7ferlaLE0yLIN7DTewY5xh4KvluhtmKCSdbh44EEnvbW38WHqTcfktcIqoOMOEaX26gMUS09FUvoZcAg+K5wNvAEfq+bm7Qs+Iws18toan6UB8Bu6ecv57Xp8Y1A2zOnrsjsdSZyuLnWFWi2HnZIy9bavnt6gm9e9vhDSU0KIUDTObArzKDGE4rDXLPzx/pxMzLYRpyZSGJAxFKB3XdkO7UOLbA4YwMlEDNc8hAVu9gHzcYdXraQIV+wAnUfvhMphbRwwUXjw8OlAISSobmEcIpDCSgV8IGY4PeDK04ZwwoonFwCxtkxvYFifs4odLhg0ZCIlu9thbUE50U7xtPHjRHQjl8eL1TPy4s06Kvly6do1zsRVzsie/Qm59Uw4LMKkqXrSoV3tCUSEIA+HruyFeAIhsB4ELuH+VmyCdUhkqFXPS0btlTV9LVNZLMVBRn8oz7zK/dtJuFF7CMwSh82qyaglwULvRfFYOSJ9qqupJtr821VatBJrX7DhxidpZBHUrxQ2kHdeUbIfRQHMBLkq8P7j7JfHq9Y5vxz1kqcKTnVJ2UxlPfEuzrzp9zFNXdFKbhBXMUMZhxbv1FHlOe79A1Zj5+YCqmQvipiieZQqajPNI8Ln0s3TsDBzfTi0HwvPjqIkyNXNwZs7/t8NVFwtyyI/SYqmCvVdlOfU7+WJoQYYjBUBCAuwXmrvQdYaB9W7uzjO8aeyj9ovIAWNJQwCl0ole7YO0YoFLDXaBYjspy2l0jamV4gevD87rO+/dnKBpC3LfPqyOSiVUmsY2H/dWsBxG2+zuD/+pK2CQBbs4Ia+TP9xeSwVvqZjjxsE4GiMyo8fzzUel+aAQ6YlQU2StDeYpjywQY5IyyRR5ja/ApGUXcuFYPb7oB7sRCfPtPYgLFFKsy6ZK0l7nVg3RJEwYjFlOUTKrFvnUfZlNNOHSy9y9kTlEQGfzjBX2H7ZnQhdxRkTC9s9zkJXmxkIAcNi4spftdRa+yrdooSgfFp1BOd0YcEGdwQgTcfWygJ8dV0wcwvzUeOnpY8OPlaoxor0Ek7sXUBnZyAOZMKfZ2HYDVtfZ27GByjRbn26y1N+v/vEtBFGvfdqo3GhJzr6U6TOT3SZfZZ5o2bOOVVqpz/MbGl1+kOft3DLyfD97CTZMmSgpWzKzfSUVp/GkzklNasUVWoHRcHXZxV4YMJ1Tzln5PRQvkj+Ce9iSHy+ciMsoR5ADRbMJUwZxaKuc+9CNTGP8A+7H3RAaf2f9I3R4P8YlwpWz3agiwKzkPatCT0uqdjInJV985Hsx5xOezOsD3L9FnRehWumbXGwFOqA5Z2JIhfPLTFFFYlNd+fyHbKRpEQZH10EHUe6VcnN/RkZrU+vDhrR7VDcdtaUUmqbtnVCjjrCaLY3GS/pQOT0xdMuBDIpvpj5eeVTDgI1gG5o9JXGV981yOFBkCGtgiQkYRDkjUymQT3X2wV3JcZZSyIGJNMkyrsg6cO82xSou8P18k5lh5uOsEs0rfOjn8hURQeGkR01BVqkOqXBByuLlYgmtB45W8qeT7btCemJjx4wD+Td4hY+CiUZreJe2UOQ/oDaU9mxuxzZ7jwItKtnfoH2OUIYgmwVjY8Eva0rpjF0Ci3JBB+Km3yOXbAULsyUrkC59Pc6f2BKKEGy2WxJy5NkLvQk89+rG0yf1kYzbWZHzk5eu3pNGH/IRMzF9Cb1Rn3zGCuu7KSsjtJCbKpER8K1WWamitrl/NlkzsDOwCmpyD4nNXwJicLlviZTUiOuQR70JhugxLxrEgPmct4JBAUtKKtikA6au4DHbDjUmVlNBSSyYmCSM939h8X2KQJm0GIBIqLyYOOS5AO4m9MgFr+2K4x6PSwMvmYOmTKsHWyiKhpe28OICa82UqJZRudCbLus5fE1h21DNq8b3IyNhRSEfVwjX8R9XgOO2sbqK3coxytP18NMf5U77z79Qb/NJxy52cB4QaO4FFMpycUUFnSVwOAGJUcNkmADJXMeOulG+ONv1o7KGLXq9kY9bAW11ckYxV56MMfa1uWAyKfxo7woNd9FewKaY8dJNSjG0NMlveaIQKW7PzgS4G27M0sFzy7T3ZSJjQlA0xSCpfqmRoVNTO+9Gtj0yrcePFVVHUyrnSZGKCIbZgFyUoRADO5XHcRty71xyIwm6kLDCdBu4fANBCkLhLtcz5J4IFhiBx+GwzFanEVVnWpf37iUsi8gbKxZOXhMTQmRz6BuUAkP8tyV6/0+oAFsEVcVgdIw9Bj9zwS9aB+Tk8O/t5Dagl7ocNIyEpXNLS3sjcBPQDSzBTbUFeuulAqpIVDL6RcHRFtFBvjUeCyKTxvTaBf+hibJSRJarFnqlZgTC18sDDxyL6q/bD+j9m2AGXbKg2tQWM+6xyrohuG6rdWYEkZvM8t5cwFxtuNKHYSOfj1qTupVg7WkVt0y6cwg3A2eHN08IYaOqkqbemnIirXBGIzMbm3Fbp8h5K1DEGMgrqt/rIWivO73dy4o8qB5DTu1jnYf0P4IcksSPUYUQBGMw+S1wwlNGvJs4PBaUSO63AcnNzvzrtJN2X+JBkZQbppuMoIzmrG6JcIiUWPIFNJlmLXLm1XRlFLziQS624/wCh4lKlLJA/RQm/LZFMY6GJaq32kR6N1pc0Htprqh206hEC0wNu9b2pVWbUVsEkBZdv7IucQ7Idt4apRY52t96kgyGcZOrMByrKnv3ndtYlP7lf/ahBAagtxiw/DzYk8K7A1nH5r77mi1nFULiw8Gmdm0HXZLzf3D53y3pzsBKz/3InYN/5bn05S0wW10IsgvWxrQxYu76M1rDOggCi4tApsWa/JYx90CAdbpNfJJGWFpYPDbLd83oiH/3z1FRdCjgcH2Jn3Ydw04yV3obYlcY6oZGLfCqWZ/Ei1Q2P5u915HUjB5jD8NorPdaASJHPHK0fPAQ0CL5tOV680cov8QLArYFphyJOTF1GpOpbvhdEOHabP4hldB6+ugYZKQU/KtJHl+JQbMVnJlGiFL9Haf+HAhbB0hHeXHMZtmOy/Zqaw23e4mk9kQ4uwZrydwB05bFJWlCWnFDg5EtuSfWA6DgR52NC7EpedFFuRnm6Db4WYKCUdGpGbC5s2s2c63OIR0/mfV1ezhlE8K/tdXurHTVvh5z4qQ+KQwY+FdLyVU1mbRaBOqoHsPAn4LL2vO74zDVSa2gl2MxXJwo0idnQXBr0KLABrioyGlZlaPpI26RIuMEvpfsZf+xUmQ96aES/G2+FTb1tml+8Phf34EZ4qrDp337IQqbmdwhIqajdJQqAkPbpz7AfJLrDTEmoVO5ehQ9bviz/o4wO5ynRIQGYA6ZAH6o6jC8pOYXp+ACH94EfHR3guYLZ0gPJDfff7WB8AEsfbg/GwU38Ueh4HNaky/o4VbE5vuIqv1UI2RvakbXvcgS+hFTYM1h3zdoMUR2sQ0oMz16B1CIKiVdvnH3JMRwy1qTgPMx5jQHIZMclsxiZ5+4rvh0ce7UVQHloLBXM9bvEqmMlssuttFgOhqWptSsC1H08DheWX8Pem8omn2TANdJBzEwCi/lWnWNjtcJko8oalnk/9oSnSaYHeTOSQWxZOyu9k6D0k41evIKot7rBu9kEjpnxT7N4M9PHnYEInLyYyZOYiqiDPVRWmFYauE1leIjbtkpqaz8p3/UWyU//MTJel2R/iSe5FOpgR62F9zV5UWcdQhDRBPamKvd4cgD1ZeuyAcjuiU9AqwA4YHd9Ahji230mB6rx3Hf3SrGpXOLrM4TvaUTO83LHdphehl82rdJxOwC/8wAsVXzZ7HtRdtY60fZQjFW88F1nB4WJcKbGRQ2Bkwr0xCpzDvRH7zpv4ZxcN4UvAArEkA2qQGqNGL7SoSds8RDTNY6aALO1muu/+A+ygsnuP1wkXAAAAAElFTkSuQmCC","dimensions":{"width":256,"height":256}}}}]},"measured":{"width":301,"height":211}}],"edges":[{"id":"e0","source":"0","sourceHandle":"out0","target":"1","targetHandle":"in0","selected":false}]}

---
## Step 2 — Connect the client

`ChaldeneClient()` opens a comm channel to the extension and starts tracking
which VP cells are ready. Run this cell after the VP canvas above is visible.

In [3]:
from chaldene import ChaldeneClient

client = ChaldeneClient()
print('Chaldene client ready.')


Chaldene client ready.


---
## Step 3 — Slider

Drag the **lower bound** knob. On release the VP cell re-executes with the
new threshold range via `client.set_input` + `client.run`.

In [4]:
import ipywidgets as widgets

@widgets.interact(
    lower=widgets.FloatSlider(
        min=0.0, max=1.00, step=0.1, value=0.5,
        description='lower:',
        continuous_update=True,
    )
)
def on_threshold_change(lower):
    ids = client.get_ready_cell_ids()
    if not ids:
        return
    client.set_input(ids[-1], '1', 'in1', [lower, 0.9])
    client.run(ids[-1])

interactive(children=(FloatSlider(value=0.5, description='lower:', max=1.0), Output()), _dom_classes=('widget-…